# 🏙️ House Price Prediction — ML Project
**Internship Submission**  
Best Model: Gradient Boosting Regressor  
Dataset: Synthetic Indian Real Estate Data (Bangalore)  
Author: [Your Name]


In [ ]:
# ── 1. Imports ────────────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import warnings, pickle, json, os

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid')
print("✅ All libraries imported successfully")


## 1. Dataset Generation
We generate a realistic synthetic dataset representing Bangalore's property market (2,500 records).

In [ ]:
# ── 2. Generate Synthetic Dataset ────────────────────────────
np.random.seed(99)
N = 2500

neighborhoods = [
    'Whitefield', 'Koramangala', 'Indiranagar', 'HSR Layout',
    'Electronic City', 'Marathahalli', 'JP Nagar', 'Jayanagar',
    'Sarjapur Road', 'Hebbal'
]

location_multiplier = {
    'Koramangala':   1.85,
    'Indiranagar':   1.75,
    'Jayanagar':     1.55,
    'HSR Layout':    1.45,
    'Whitefield':    1.35,
    'Marathahalli':  1.25,
    'Hebbal':        1.20,
    'JP Nagar':      1.15,
    'Sarjapur Road': 1.10,
    'Electronic City': 1.00,
}

prop_types     = ['Apartment', 'Independent House', 'Villa', 'Studio']
furnish_opts   = ['Unfurnished', 'Semi-furnished', 'Fully Furnished']
condition_opts = ['Excellent', 'Good', 'Fair', 'Needs Renovation']

data = []
for _ in range(N):
    nbr   = np.random.choice(neighborhoods)
    ptype = np.random.choice(prop_types, p=[0.50, 0.25, 0.15, 0.10])
    sqft  = int(np.random.normal(1400, 550))
    sqft  = max(300, min(sqft, 8000))
    beds  = np.random.choice([1,2,3,4,5], p=[0.10,0.30,0.35,0.20,0.05])
    baths = max(1, beds - np.random.choice([0,1]))
    age   = max(0, int(np.random.exponential(6)))
    park  = np.random.choice([0,1,2], p=[0.20,0.50,0.30])
    floor = np.random.randint(0, 25) if ptype == 'Apartment' else 0
    furnish   = np.random.choice(furnish_opts)
    condition = np.random.choice(condition_opts, p=[0.20,0.50,0.20,0.10])
    # New: amenity features
    has_balcony = int(np.random.choice([0,1], p=[0.35, 0.65]))
    lift        = int(np.random.choice([0,1], p=[0.30, 0.70]) if ptype == 'Apartment' else 0)

    base_price = 3800 * sqft
    base_price *= location_multiplier[nbr]
    base_price *= (1 + 0.14 * beds)
    base_price *= (1 + 0.07 * park)
    base_price *= max(0.6, 1 - 0.015 * age)
    if furnish == 'Semi-furnished':     base_price *= 1.07
    if furnish == 'Fully Furnished':    base_price *= 1.16
    if condition == 'Excellent':        base_price *= 1.10
    if condition == 'Needs Renovation': base_price *= 0.82
    if ptype == 'Villa':                base_price *= 1.35
    if ptype == 'Studio':               base_price *= 0.78
    if has_balcony:                     base_price *= 1.03
    if lift:                            base_price *= 1.02
    noise = np.random.normal(1.0, 0.055)
    price = int(base_price * noise / 100000) * 100000

    data.append({
        'Neighborhood': nbr, 'Property_Type': ptype,
        'Area_sqft': sqft, 'Bedrooms': beds, 'Bathrooms': baths,
        'Age_years': age, 'Parking_spots': park, 'Floor': floor,
        'Has_Balcony': has_balcony, 'Lift': lift,
        'Furnishing': furnish, 'Condition': condition, 'Price': price
    })

df = pd.DataFrame(data)
print(f"Dataset shape: {df.shape}")
print(f"Price range: ₹{df['Price'].min():,.0f} — ₹{df['Price'].max():,.0f}")
df.head()


## 2. Exploratory Data Analysis (EDA)

In [ ]:
# ── 3. EDA ───────────────────────────────────────────────────
print("=== Dataset Info ===")
print(df.dtypes)
print("\n=== Missing Values ===")
print(df.isnull().sum())
print("\n=== Basic Stats ===")
df[['Area_sqft','Bedrooms','Age_years','Price']].describe()


In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
fig.suptitle('EDA — Bangalore Property Market Dataset', fontsize=15, y=1.01)

# Price distribution
axes[0,0].hist(df['Price']/1e6, bins=40, color='#0d9488', edgecolor='white')
axes[0,0].set_title('Price Distribution (₹ Millions)')
axes[0,0].set_xlabel('Price (₹ M)')

# Area vs Price
axes[0,1].scatter(df['Area_sqft'], df['Price']/1e6, alpha=0.3, s=10, color='#f59e0b')
axes[0,1].set_title('Area vs Price')
axes[0,1].set_xlabel('Area (sq ft)'); axes[0,1].set_ylabel('Price (₹ M)')

# Avg price by neighborhood
nbr_avg = df.groupby('Neighborhood')['Price'].mean().sort_values()
axes[0,2].barh(nbr_avg.index, nbr_avg.values/1e6, color='#0d9488')
axes[0,2].set_title('Avg Price by Neighborhood (₹ M)')

# Bedrooms vs Price
df.groupby('Bedrooms')['Price'].mean().div(1e6).plot(kind='bar', ax=axes[1,0], color='#6366f1')
axes[1,0].set_title('Avg Price by Bedrooms'); axes[1,0].set_xlabel('Bedrooms')

# Age vs Price
axes[1,1].scatter(df['Age_years'], df['Price']/1e6, alpha=0.3, s=10, color='#ec4899')
axes[1,1].set_title('Age vs Price'); axes[1,1].set_xlabel('Age (years)')

# Furnishing vs Price
df.groupby('Furnishing')['Price'].mean().div(1e6).plot(kind='bar', ax=axes[1,2], color='#8b5cf6')
axes[1,2].set_title('Avg Price by Furnishing')

plt.tight_layout()
plt.savefig('eda_plots.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ EDA plots saved as eda_plots.png")


In [ ]:
plt.figure(figsize=(9, 7))
corr = df[['Area_sqft','Bedrooms','Bathrooms','Age_years','Parking_spots',
           'Floor','Has_Balcony','Lift','Price']].corr()
sns.heatmap(corr, annot=True, fmt='.2f', cmap='YlOrRd', linewidths=0.5)
plt.title('Correlation Heatmap')
plt.tight_layout()
plt.savefig('correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()


## 3. Data Preprocessing & Feature Engineering

In [ ]:
# ── 4. Preprocessing ─────────────────────────────────────────
df_model = df.copy()

# Feature engineering
df_model['Room_ratio']  = df_model['Bathrooms'] / df_model['Bedrooms']
df_model['Total_rooms'] = df_model['Bedrooms']  + df_model['Bathrooms']

# Label-encode categoricals
cat_cols = ['Neighborhood', 'Property_Type', 'Furnishing', 'Condition']
encoders = {}
for col in cat_cols:
    le = LabelEncoder()
    df_model[col + '_enc'] = le.fit_transform(df_model[col])
    encoders[col] = le
    print(f"Encoded {col}: {list(le.classes_)}")

# Feature matrix (14 features — includes Has_Balcony & Lift)
feature_cols = [
    'Area_sqft', 'Bedrooms', 'Bathrooms', 'Age_years',
    'Parking_spots', 'Floor', 'Has_Balcony', 'Lift',
    'Room_ratio', 'Total_rooms',
    'Neighborhood_enc', 'Property_Type_enc', 'Furnishing_enc', 'Condition_enc'
]

X = df_model[feature_cols]
y = df_model['Price']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=99)
print(f"\nTrain: {X_train.shape}, Test: {X_test.shape}")


## 4. Model Training & Comparison

In [ ]:
# ── 5. Train Models ───────────────────────────────────────────
models = {
    'Linear Regression': LinearRegression(),
    'Decision Tree':     DecisionTreeRegressor(max_depth=10, random_state=99),
    'Random Forest':     RandomForestRegressor(n_estimators=150, random_state=99, n_jobs=-1),
    'Gradient Boosting': GradientBoostingRegressor(n_estimators=150, learning_rate=0.08, random_state=99),
}

results = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    rmse  = np.sqrt(mean_squared_error(y_test, preds))
    mae   = mean_absolute_error(y_test, preds)
    r2    = r2_score(y_test, preds)
    mape  = np.mean(np.abs((y_test - preds) / y_test)) * 100
    results[name] = {'RMSE': rmse, 'MAE': mae, 'R²': r2, 'MAPE(%)': mape}
    print(f"{name:25s} | RMSE: ₹{rmse:>12,.0f} | R²: {r2:.4f} | MAPE: {mape:.2f}%")

results_df = pd.DataFrame(results).T
results_df


In [ ]:
# ── 6. Cross-Validation (best model: Gradient Boosting) ───────
best_model = models['Gradient Boosting']

cv_scores = cross_val_score(best_model, X, y, cv=5, scoring='r2')
print(f"Cross-Validation R² Scores (5-fold): {cv_scores}")
print(f"Mean CV R²: {cv_scores.mean():.4f}  |  Std: {cv_scores.std():.4f}")
print(f"\n→ Gradient Boosting selected as best model (highest R² + stable CV)")


In [ ]:
# ── 7. Feature Importance (Gradient Boosting) ─────────────────
importances = pd.Series(best_model.feature_importances_, index=feature_cols).sort_values(ascending=True)

plt.figure(figsize=(9, 5))
importances.plot(kind='barh', color='#0d9488', edgecolor='white')
plt.title('Feature Importances — Gradient Boosting')
plt.xlabel('Importance Score')
plt.tight_layout()
plt.savefig('feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# ── 8. Actual vs Predicted Plot ───────────────────────────────
preds_test = best_model.predict(X_test)

plt.figure(figsize=(8, 6))
plt.scatter(y_test/1e6, preds_test/1e6, alpha=0.4, s=15, color='#0d9488')
mn, mx = (y_test/1e6).min(), (y_test/1e6).max()
plt.plot([mn, mx], [mn, mx], 'r--', lw=1.5, label='Perfect prediction')
plt.xlabel('Actual Price (₹ M)'); plt.ylabel('Predicted Price (₹ M)')
plt.title('Actual vs Predicted — Gradient Boosting')
plt.legend()
plt.tight_layout()
plt.savefig('actual_vs_predicted.png', dpi=150, bbox_inches='tight')
plt.show()

r2 = r2_score(y_test, preds_test)
print(f"Final Model R² Score: {r2:.4f}  ({r2*100:.1f}% variance explained)")


## 5. Save Model & Metadata for API

In [ ]:
# ── 9. Save Model & Encoder Metadata ─────────────────────────
os.makedirs('model', exist_ok=True)

with open('model/house_price_model.pkl', 'wb') as f:
    pickle.dump(best_model, f)

with open('model/encoders.pkl', 'wb') as f:
    pickle.dump(encoders, f)

metadata = {
    'feature_cols': feature_cols,
    'neighborhoods': list(encoders['Neighborhood'].classes_),
    'property_types': list(encoders['Property_Type'].classes_),
    'furnishings':    list(encoders['Furnishing'].classes_),
    'conditions':     list(encoders['Condition'].classes_),
    'model_r2':       round(float(r2_score(y_test, best_model.predict(X_test))), 4),
    'cv_r2_mean':     round(float(cv_scores.mean()), 4),
    'cv_r2_std':      round(float(cv_scores.std()), 4),
}
with open('model/metadata.json', 'w') as f:
    json.dump(metadata, f, indent=2)

print("✅ Saved:")
print("   model/house_price_model.pkl")
print("   model/encoders.pkl")
print("   model/metadata.json")
print(f"\nModel R²: {metadata['model_r2']} — ready for API deployment")


## 6. Sample Prediction

In [ ]:
# ── 10. Sample Prediction ────────────────────────────────────
sample = {
    'Area_sqft': 1200,
    'Bedrooms': 2,
    'Bathrooms': 2,
    'Age_years': 3,
    'Parking_spots': 1,
    'Floor': 7,
    'Has_Balcony': 1,
    'Lift': 1,
    'Neighborhood': 'Whitefield',
    'Property_Type': 'Apartment',
    'Furnishing': 'Semi-furnished',
    'Condition': 'Good'
}

sample_enc = sample.copy()
for col in ['Neighborhood', 'Property_Type', 'Furnishing', 'Condition']:
    sample_enc[col + '_enc'] = encoders[col].transform([sample[col]])[0]

sample_enc['Room_ratio']  = sample_enc['Bathrooms'] / sample_enc['Bedrooms']
sample_enc['Total_rooms'] = sample_enc['Bedrooms']  + sample_enc['Bathrooms']

X_sample = pd.DataFrame([[sample_enc[fc] for fc in feature_cols]], columns=feature_cols)
predicted_price = best_model.predict(X_sample)[0]

print(f"Sample Property: {sample['Bedrooms']}BHK {sample['Property_Type']} in {sample['Neighborhood']}")
print(f"Area: {sample['Area_sqft']} sq ft | Age: {sample['Age_years']} yrs | {sample['Furnishing']}")
print(f"Balcony: {'Yes' if sample['Has_Balcony'] else 'No'} | Lift: {'Yes' if sample['Lift'] else 'No'}")
print(f"\n{'='*45}")
print(f"  Predicted Price: ₹{predicted_price:,.0f}")
print(f"  (~₹{predicted_price/1e7:.2f} Cr)")
print(f"{'='*45}")
